In [1]:
import itertools
import json
LOGIC_RULES = {
    '→':  {('T','T'), ('F','T'), ('F','F')},  
    '←':  {('T','T'), ('T','F'), ('F','F')},  
    '→←': {('T','T'), ('F','F')},             
    '×':  {('T','F'), ('F','T')}             
}

def _forward_propagate_from(edges, i, domains):
    for j in range(i, len(edges)):
        u, t, v = edges[j]
        R = LOGIC_RULES[t]

        allowed_pairs = [(a, b) for (a, b) in R if a in domains[u] and b in domains[v]]
        if not allowed_pairs:
            return False

        new_u = {a for (a, _) in allowed_pairs}
        new_v = {b for (_, b) in allowed_pairs}
        if new_u != domains[u]:
            domains[u] = new_u
        if new_v != domains[v]:
            domains[v] = new_v

        if not domains[u] or not domains[v]:
            return False
    return True

def _dfs_edges(edges, i, domains):

    if i == len(edges):
        return True

    u, t, v = edges[i]
    R = LOGIC_RULES[t]

    candidates = [(a, b) for (a, b) in R if a in domains[u] and b in domains[v]]
    if not candidates:
        return False

    for a, b in candidates:
        dom2 = {n: set(vals) for n, vals in domains.items()}
        dom2[u] = {a}
        dom2[v] = {b}

        if _forward_propagate_from(edges, i, dom2):
            if _dfs_edges(edges, i + 1, dom2):
                return True
    return False

def evaluate_path_partition_dfs(source_nodes, edges):
    all_nodes = list(dict.fromkeys(
        list(source_nodes) + [u for u,_,_ in edges] + [v for _,_,v in edges]
    ))

    valid, invalid = [], []
    for proj in itertools.product(['T','F'], repeat=len(source_nodes)):
        domains = {n: {'T','F'} for n in all_nodes}
        for n, val in zip(source_nodes, proj):
            domains[n] = {val}

        sat = _forward_propagate_from(edges, 0, domains) and _dfs_edges(edges, 0, domains)
        (valid if sat else invalid).append(list(proj))

    return valid, invalid

def process_path_file(path_file, out_file):
    with open(path_file, "r") as f_in, open(out_file, "w") as f_out:
        s = 0
        for line in f_in:
            s += 1
            print(s)

            rec = json.loads(line)
            source_nodes = rec["source_nodes"]
            edges = rec["edges"]

            print(len(source_nodes))
            print(len(edges))

            valid_sets, invalid_sets = evaluate_path_partition_dfs(source_nodes, edges)

            rec["valid_sets"] = valid_sets
            rec["invalid_sets"] = invalid_sets
            print("valid",valid_sets)
            print("invalid",invalid_sets)

            f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"Processed {s} records, saved to {out_file}")

In [2]:
import itertools
import json

LOGIC_RULES = {
    '→':  {('T','T'), ('F','T'), ('F','F')},
    '←':  {('T','T'), ('T','F'), ('F','F')},
    '→←': {('T','T'), ('F','F')},
    '×':  {('T','F'), ('F','T')}
}

def _forward_propagate_from(edges, i, domains):
    for j in range(i, len(edges)):
        u, t, v = edges[j]
        R = LOGIC_RULES[t]
        allowed_pairs = [(a, b) for (a, b) in R if a in domains[u] and b in domains[v]]
        if not allowed_pairs:
            return False
        new_u = {a for (a, _) in allowed_pairs}
        new_v = {b for (_, b) in allowed_pairs}
        if new_u != domains[u]:
            domains[u] = new_u
        if new_v != domains[v]:
            domains[v] = new_v
        if not domains[u] or not domains[v]:
            return False
    return True

def _dfs_edges(edges, i, domains):
    if i == len(edges):
        return True
    u, t, v = edges[i]
    R = LOGIC_RULES[t]
    candidates = [(a, b) for (a, b) in R if a in domains[u] and b in domains[v]]
    if not candidates:
        return False
    for a, b in candidates:
        dom2 = {n: set(vals) for n, vals in domains.items()}
        dom2[u] = {a}
        dom2[v] = {b}
        if _forward_propagate_from(edges, i, dom2):
            if _dfs_edges(edges, i + 1, dom2):
                return True
    return False

def evaluate_path_partition_dfs(source_nodes, edges):
    all_nodes = list(dict.fromkeys(
        list(source_nodes) + [u for u,_,_ in edges] + [v for _,_,v in edges]
    ))
    valid, invalid = [], []
    for proj in itertools.product(['T','F'], repeat=len(source_nodes)):
        # Check consistency: same node must get same value
        assignment = {}
        consistent = True
        for n, val in zip(source_nodes, proj):
            if n in assignment:
                if assignment[n] != val:
                    consistent = False
                    break
            else:
                assignment[n] = val

        if not consistent:
            invalid.append(list(proj))
            continue

        domains = {n: {'T','F'} for n in all_nodes}
        for n, val in assignment.items():
            domains[n] = {val}
        sat = _forward_propagate_from(edges, 0, domains) and _dfs_edges(edges, 0, domains)
        (valid if sat else invalid).append(list(proj))
    return valid, invalid

def process_path_file(path_file, out_file):
    with open(path_file, "r") as f_in, open(out_file, "w") as f_out:
        s = 0
        for line in f_in:
            s += 1
            print(s)
            rec = json.loads(line)
            source_nodes = rec["source_nodes"]
            edges = rec["edges"]
            print(len(source_nodes))
            print(len(edges))
            valid_sets, invalid_sets = evaluate_path_partition_dfs(source_nodes, edges)
            rec["valid_sets"] = valid_sets
            rec["invalid_sets"] = invalid_sets
            print("valid",valid_sets)
            print("invalid",invalid_sets)
            f_out.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"Processed {s} records, saved to {out_file}")

In [ ]:
process_path_file("rewrite_paths5.dedup_wn.jsonl", "rewrite_paths5.dedup_wn_processed.jsonl")

1
5
11
valid [['T', 'F', 'T', 'F', 'T'], ['T', 'F', 'F', 'F', 'T'], ['F', 'T', 'T', 'T', 'T'], ['F', 'T', 'T', 'F', 'T'], ['F', 'T', 'F', 'T', 'T'], ['F', 'T', 'F', 'T', 'F'], ['F', 'T', 'F', 'F', 'T'], ['F', 'T', 'F', 'F', 'F'], ['F', 'F', 'T', 'F', 'T'], ['F', 'F', 'F', 'F', 'T']]
invalid [['T', 'T', 'T', 'T', 'T'], ['T', 'T', 'T', 'T', 'F'], ['T', 'T', 'T', 'F', 'T'], ['T', 'T', 'T', 'F', 'F'], ['T', 'T', 'F', 'T', 'T'], ['T', 'T', 'F', 'T', 'F'], ['T', 'T', 'F', 'F', 'T'], ['T', 'T', 'F', 'F', 'F'], ['T', 'F', 'T', 'T', 'T'], ['T', 'F', 'T', 'T', 'F'], ['T', 'F', 'T', 'F', 'F'], ['T', 'F', 'F', 'T', 'T'], ['T', 'F', 'F', 'T', 'F'], ['T', 'F', 'F', 'F', 'F'], ['F', 'T', 'T', 'T', 'F'], ['F', 'T', 'T', 'F', 'F'], ['F', 'F', 'T', 'T', 'T'], ['F', 'F', 'T', 'T', 'F'], ['F', 'F', 'T', 'F', 'F'], ['F', 'F', 'F', 'T', 'T'], ['F', 'F', 'F', 'T', 'F'], ['F', 'F', 'F', 'F', 'F']]
2
5
11
valid [['T', 'F', 'T', 'F', 'T'], ['T', 'F', 'F', 'F', 'T'], ['F', 'T', 'T', 'T', 'T'], ['F', 'T', 'T', 'F

In [6]:
import json

with open("rewrite_paths3.dedup_wn_processed.jsonl") as f1, \
     open("rewrite_paths3.dedup_wn.jsonl") as f2:
    for i, (l1, l2) in enumerate(zip(f1, f2)):
        if 35>i >= 20:
            break
        r1, r2 = json.loads(l1), json.loads(l2)
        # 比较除 valid_sets/invalid_sets 外的字段是否一致
        shared_keys = set(r2.keys())
        match = all(r1.get(k) == r2.get(k) for k in shared_keys)
        extra_keys = set(r1.keys()) - shared_keys
        print(f"Line {i+1}: base fields match={match}, extra keys in processed={extra_keys}")

Line 1: base fields match=True, extra keys in processed=set()
Line 2: base fields match=False, extra keys in processed=set()
Line 3: base fields match=True, extra keys in processed=set()
Line 4: base fields match=True, extra keys in processed=set()
Line 5: base fields match=True, extra keys in processed=set()
Line 6: base fields match=True, extra keys in processed=set()
Line 7: base fields match=True, extra keys in processed=set()
Line 8: base fields match=True, extra keys in processed=set()
Line 9: base fields match=True, extra keys in processed=set()
Line 10: base fields match=True, extra keys in processed=set()
Line 11: base fields match=True, extra keys in processed=set()
Line 12: base fields match=True, extra keys in processed=set()
Line 13: base fields match=True, extra keys in processed=set()
Line 14: base fields match=True, extra keys in processed=set()
Line 15: base fields match=True, extra keys in processed=set()
Line 16: base fields match=True, extra keys in processed=set()
